# Aşama 9: Enerji terimi

Şimdiye kadar her olayın sadece **yönünü** kullandık. Ama elimizde bir ipucu daha var: **enerji**.

- Atmosferik (arka plan) nötrinoların enerji spektrumu çok dik: yüksek enerjili olanlar çok nadir.
- Kozmik hızlandırıcılardan gelen nötrinoların spektrumu daha yatık: yüksek enerjiler görece daha sık.

Yani **yüksek enerjili bir olay, kaynaktan gelmiş olmaya daha yatkın**. Likelihood'a bu bilgiyi ekliyoruz:

$$\frac{S_i}{B_i} = \underbrace{\frac{S_{\text{uzay}}}{B_{\text{uzay}}}}_{\text{Aşama 3}} \times \underbrace{\frac{P_{\text{sinyal}}(E_i \mid \gamma)}{P_{\text{arka}}(E_i)}}_{\text{yeni}}$$

Kaynağın spektrum eğimi γ'yı da veriden fit ediyoruz: kaynak hakkında yeni bir fiziksel ölçüm.

In [ ]:
import sys; sys.path.append('../src')
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import icecube_ps as ps
importlib.reload(ps)
assert float(ps.SURUM) >= float('9.0'), f"src/icecube_ps.py eski sürüm ({getattr(ps, 'SURUM', 'yok')}). Yenisini kopyala, kernel'i yeniden başlat."
try:
    from tqdm.auto import tqdm          # ilerleme çubuğu
except ImportError:
    tqdm = None

pd.set_option('display.float_format', '{:.3g}'.format)
DATA, RESULTS = Path('../data'), Path('../results')
ev = ps.veri_yukle(DATA)
B = ps.ArkaPlan(ev['dec'].values)
print('icecube_ps', ps.SURUM, '| olay:', f'{len(ev):,}')
from scipy import stats

## 1. Sinyalin enerji dağılımı nereden geliyor?

Arka planınki kolay: verinin kendisi. Sinyalinki için dedektör simülasyon tabloları lazım: "**gerçek** enerjisi E olan bir nötrino, dedektörde **hangi** enerjiyle ölçülür?" Bu tabloya *smearing matrisi* deniyor. Bir $E^{-\gamma}$ akısını önce etkin alanla, sonra bu tabloyla "dedektörden geçirince" ölçülen enerji dağılımını elde ediyoruz.

In [ ]:
alanlar = ps.etkin_alanlari_yukle(DATA)
sezon, smearing = ps.smearing_yukle(DATA)
print(f'Smearing sezonu: {sezon}, {len(smearing):,} satır')
ornek = next(ps._metin_dosyalari(DATA, 'irfs/*smearing*'))
print(open(ornek, encoding='latin-1').read()[:350])     # biçim kontrolü

EP = ps.EnerjiPDF(ev, alanlar[ps._irf_sezonu(sezon, alanlar)], smearing)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
x = 0.5 * (EP.logE_kenar[1:] + EP.logE_kenar[:-1])
for ax, dec, baslik in [(axs[0], 0.0, 'Ekvator (NGC 1068 yönü)'), (axs[1], 60.0, 'Kuzey (GB6 J1542 yönü)')]:
    i, _ = EP._kutu(np.array([dec]), np.array([3.0]))
    ax.step(x, EP.P_arka[i[0]], where='mid', color='#333', lw=2.5, label='Arka plan (veri)')
    for g, renk in [(2.0, '#d1495b'), (2.5, '#edae49'), (3.0, '#66a182'), (3.5, '#4c9be8')]:
        k = int(np.argmin(np.abs(EP.gamma - g)))
        ax.step(x, EP.P_sinyal[k, i[0]], where='mid', color=renk, label=f'Sinyal γ = {g}')
    ax.set_yscale('log'); ax.set_ylim(1e-5, 1)
    ax.set_xlabel('ölçülen log₁₀(E / GeV)'); ax.set_title(baslik); ax.grid(alpha=0.3)
axs[0].set_ylabel('olasılık'); axs[0].legend(fontsize=9)
plt.tight_layout()
plt.savefig(RESULTS / 'asama9_enerji_pdf.png', dpi=150, bbox_inches='tight')
plt.show()

**Grafiği okurken:** Siyah çizgi (arka plan) yüksek enerjiye doğru hızla düşüyor. Renkli çizgiler (sinyal) sağa kayık, özellikle γ = 2. Sağ taraftaki olaylar "sinyal gibi" görünüyor: enerji terimi tam olarak bu farkı kullanıyor.

## 2. Dört kaynak: enerjili fit

Her kaynak için nₛ ve γ birlikte fit ediliyor. Karşılaştırma için Aşama 3'teki (sadece yön) sonuçları yanına koyuyoruz.

In [ ]:
asama3 = pd.read_csv(RESULTS / 'asama3_likelihood.csv').set_index('kaynak')
modeller, fitler = {}, []
for ad, (rk, dk) in ps.KAYNAKLAR.items():
    L = ps.EnerjiliLikelihood(ev, B, EP, rk, dk)
    L.sinyal_havuzu_ayarla(ev)
    modeller[ad] = L
    ns, ts, gam = L.fit_enerji()
    fitler.append(dict(kaynak=ad, ns=ns, gamma=gam, TS=ts, TS_sadece_yon=asama3.loc[ad, 'TS']))
pd.DataFrame(fitler).set_index('kaynak')

## 3. Anlamlılık

Yine RA karıştırma: olayların enerjilerine ve Dec'lerine dokunmuyoruz, sadece yönlerini dağıtıyoruz. ⏱️ Kaynak başına 2000 deneme.

γ'yı da fit ettiğimiz için Wilks'in basit formu artık tam uymuyor. Kuyruk için χ² dağılımının serbestlik derecesini karıştırılmış denemelerden tahmin ediyoruz (χ²'nin ortalaması serbestlik derecesine eşittir).

In [ ]:
for f in fitler:
    ad = f['kaynak']
    dosya = RESULTS / f"ts_arka_enerji_{ad.replace(' ', '_').replace('+', 'p')}.npy"
    if dosya.exists():
        ts_a = np.load(dosya)
    else:
        ts_a = modeller[ad].arka_plan_denemeleri_enerji(2000, np.random.default_rng(90), ilerleme=tqdm)
        np.save(dosya, ts_a)
    eta, sd = np.mean(ts_a > 0), np.mean(ts_a[ts_a > 0])
    p_mc = (np.sum(ts_a >= f['TS']) + 1) / (len(ts_a) + 1)
    p_kuyruk = eta * stats.chi2.sf(f['TS'], df=sd) if f['TS'] > 0 else 1.0
    # TS = 0 (fazlalık yok) → anlamlılık 0σ olarak gösterilsin
    f.update(sigma_MC=max(0.0, ps.sigma(p_mc)), sigma_kuyruk=max(0.0, ps.sigma(p_kuyruk)),
             sigma_sadece_yon=asama3.loc[ad, 'sigma_Wilks'])

tablo9 = pd.DataFrame(fitler).set_index('kaynak')
tablo9.to_csv(RESULTS / 'asama9_enerji.csv')
tablo9[['ns', 'gamma', 'TS', 'sigma_MC', 'sigma_kuyruk', 'sigma_sadece_yon']]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(tablo9))
ax.bar(x - 0.2, tablo9['sigma_sadece_yon'], 0.4, color='#9aa5b8', label='Sadece yön (Aşama 3)')
ax.bar(x + 0.2, tablo9['sigma_kuyruk'], 0.4, color='#d1495b', label='Yön + enerji (Aşama 9)')
ax.set_xticks(x); ax.set_xticklabels(tablo9.index)
ax.set_ylabel('anlamlılık [σ]'); ax.axhline(3, color='#333', ls=':', lw=1)
ax.legend(); ax.set_title('Enerji bilgisi eklemenin etkisi')
plt.savefig(RESULTS / 'asama9_karsilastirma.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Analiz ne kadar güçlendi?

Asıl adil karşılaştırma bu: aynı sahte kaynakları hem eski (sadece yön) hem yeni (yön + enerji) yöntemle analiz edip hassasiyetleri karşılaştırıyoruz. NGC 1068'in yönünde, γ = 2 spektrumlu sahte kaynaklarla. ⏱️ Birkaç dakika.

In [ ]:
dosya = RESULTS / 'asama9_hassasiyet_karsilastirma.npz'
L = modeller['NGC 1068']
izgara = np.arange(0, 61, 6)
if dosya.exists():
    d = np.load(dosya); arka_y, arka_e, sin_y, sin_e = d['arka_y'], d['arka_e'], d['sin_y'], d['sin_e']
else:
    rng = np.random.default_rng(99)
    arka_y = L.arka_plan_denemeleri(1000, rng)
    arka_e = L.arka_plan_denemeleri_enerji(1000, rng)
    sin_y = np.array([L.sinyal_denemeleri_enerji(n, 2.0, 100, rng, enerji_kullan=False) for n in izgara])
    sin_e = np.array([L.sinyal_denemeleri_enerji(n, 2.0, 100, rng, enerji_kullan=True) for n in izgara])
    np.savez(dosya, arka_y=arka_y, arka_e=arka_e, sin_y=sin_y, sin_e=sin_e)

ns_y, f_y = ps.esik_gecen_ns(izgara, sin_y, np.median(arka_y), 0.9)
ns_e, f_e = ps.esik_gecen_ns(izgara, sin_e, np.median(arka_e), 0.9)
print(f"Hassasiyet (γ=2 kaynak): sadece yön {ns_y:.1f} olay  →  yön + enerji {ns_e:.1f} olay")
print(f"İyileşme: {ns_y / ns_e:.1f} kat")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(izgara, f_y, 'o-', color='#9aa5b8', label=f'Sadece yön  →  {ns_y:.0f} olay')
ax.plot(izgara, f_e, 'o-', color='#d1495b', label=f'Yön + enerji  →  {ns_e:.0f} olay')
ax.axhline(0.9, color='#333', ls=':', lw=1)
ax.set_xlabel('eklenen ortalama sinyal nₛ'); ax.set_ylabel('TS > arka plan medyanı oranı')
ax.set_title('NGC 1068 yönünde hassasiyet: enerji terimi ile ve olmadan'); ax.legend(); ax.grid(alpha=0.3)
plt.savefig(RESULTS / 'asama9_hassasiyet.png', dpi=150, bbox_inches='tight')
plt.show()

## Yorumlarken

- **γ (spektrum eğimi) yeni bir fiziksel ölçüm.** IceCube NGC 1068 için oldukça **yumuşak** (γ ≈ 3 civarı) bir spektrum buldu: kaynak çoğunlukla düşük enerjili nötrino üretiyor. Bizim fit ettiğimiz γ bununla karşılaştırılabilir.
- **Enerji her kaynakta anlamlılığı artırmak zorunda değil.** Yumuşak spektrumlu bir kaynağın nötrinoları arka plana benzer enerjilerde olduğu için enerji onu ayırt etmeye pek yardım etmez. Sert spektrumlu (γ ≈ 2) kaynaklarda kazanç büyüktür; son grafikteki hassasiyet karşılaştırması bunu gösteriyor.
- **Sadeleştirmeler:** Tek sezonun simülasyon tablolarını tüm veriye uyguladık; IceCube her sezonu ayrı modelliyor. γ'yı kaba bir ızgarada taradık.